# 실습 2: Autograd와 미분 이해하기 (The Engine)

**목표:** 모델의 학습 과정을 가능하게 하는 핵심 원리인 **자동 미분(Autograd)**이 내부적으로 어떻게 작동하는지 추적하며 이해한다.

## 개념 복기 및 이론 점검
1. **미분 / 기울기(Gradient)**: 어떤 변수가 조금 변했을 때 함수 값이 얼마나 변하는가 (변화율).
2. **역전파(Backpropagation)**: 손실을 줄이기 위해 결과에서부터 거꾸로 기울기를 계산하여 가중치를 수정하는 과정.
3. `requires_grad=True`: 이 Tensor에 대한 연산을 **추적하기 시작하라**는 신호.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/2주차/lab_02_autograd_mechanism.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

---
## 1. 간단 함수 미분 실습 (개념 증명)

함수: $L = (A \cdot B^2 + 3A) \cdot B = A B^3 + 3 A B$

**수동 미분:**
- $\dfrac{\partial L}{\partial A} = B^3 + 3B$
- $\dfrac{\partial L}{\partial B} = 3 A B^2 + 3 A$

**값 대입 (A=2, B=3):**
- $\dfrac{\partial L}{\partial A} = 27 + 9 = 36$
- $\dfrac{\partial L}{\partial B} = 3 \cdot 2 \cdot 9 + 3 \cdot 2 = 54 + 6 = 60$

In [ ]:
# requires_grad=True → 이 변수는 추적 대상
A = torch.tensor(2.0, requires_grad=True)
B = torch.tensor(3.0, requires_grad=True)

# 순전파(Forward): 함수 값 계산
L = (A * B**2 + 3 * A) * B
print(f"L 값 = {L.item()}   (수동 계산: A*B^3 + 3AB = 2*27 + 18 = {2*27 + 18})")

# 역전파(Backward): dL/dA, dL/dB 계산
L.backward()

print(f"\nPyTorch가 계산한 dL/dA = {A.grad.item()}  (수동 계산 = 36)")
print(f"PyTorch가 계산한 dL/dB = {B.grad.item()}  (수동 계산 = 60)")

assert A.grad.item() == 36.0
assert B.grad.item() == 60.0
print("\n✅ 수동 미분 값과 PyTorch의 결과가 정확히 일치합니다!")

### 1-2. 계산 그래프 이해하기

`requires_grad=True`인 Tensor가 들어간 연산은 **계산 그래프(computational graph)**에 기록됩니다.  
`grad_fn` 속성이 바로 그 기록입니다.

In [ ]:
A = torch.tensor(2.0, requires_grad=True)
B = torch.tensor(3.0, requires_grad=True)

step1 = B ** 2
step2 = A * step1
step3 = step2 + 3 * A
L = step3 * B

print(f"step1 grad_fn: {step1.grad_fn}")
print(f"step2 grad_fn: {step2.grad_fn}")
print(f"step3 grad_fn: {step3.grad_fn}")
print(f"L     grad_fn: {L.grad_fn}")

### 1-3. `requires_grad=False` 이면 추적되지 않는다

In [ ]:
x = torch.tensor(2.0)  # 기본값: requires_grad=False
y = x ** 2 + 3 * x
print(f"y.grad_fn = {y.grad_fn}  ← 추적 안 됨 (None)")

try:
    y.backward()
except RuntimeError as e:
    print(f"\n❌ backward() 호출 실패: {e}")

### 1-4. `torch.no_grad()` — 추적을 일시적으로 끄기

추론(inference)할 때는 Gradient가 필요 없습니다. 메모리와 속도를 아끼기 위해 추적을 끕니다.

In [ ]:
A = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = A ** 2 + 3 * A
    print(f"no_grad 안에서 y.requires_grad = {y.requires_grad}")
    print(f"no_grad 안에서 y.grad_fn = {y.grad_fn}")

---
## 2. 훈련 과정 시뮬레이션 (최소 규모)

가상 선형 모델 $y = W x + b$ 를 학습시켜 봅니다.

- 정답 규칙: $y = 2x + 1$
- 초기 $W, b$는 랜덤 → 학습 후 $W \approx 2,\ b \approx 1$로 수렴하는지 확인

In [ ]:
# 데이터 준비: y = 2x + 1 + 약간의 노이즈
x_data = torch.linspace(-3, 3, 50).unsqueeze(1)       # (50, 1)
y_true = 2 * x_data + 1 + 0.1 * torch.randn_like(x_data)

# 학습할 파라미터 W, b (requires_grad=True)
W = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"초기 W = {W.item():.4f}, b = {b.item():.4f}")

# Optimizer: SGD (확률적 경사 하강법)
optimizer = torch.optim.SGD([W, b], lr=0.05)

loss_history = []

for epoch in range(100):
    # (1) Forward
    y_pred = W * x_data + b

    # (2) Loss (MSE)
    loss = ((y_pred - y_true) ** 2).mean()

    # (3) Backward — 기울기 계산
    optimizer.zero_grad()   # ★ 이전 단계의 grad를 0으로 초기화
    loss.backward()

    # (4) 파라미터 업데이트
    optimizer.step()

    loss_history.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss = {loss.item():.4f} | W = {W.item():.4f}, b = {b.item():.4f}")

print(f"\n🎯 학습 후 W = {W.item():.4f} (목표 2.0)")
print(f"🎯 학습 후 b = {b.item():.4f} (목표 1.0)")

In [ ]:
# Loss 곡선과 학습된 직선 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history)
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].grid(True)

with torch.no_grad():
    y_fit = W * x_data + b
axes[1].scatter(x_data.numpy(), y_true.numpy(), label="data", alpha=0.6)
axes[1].plot(x_data.numpy(), y_fit.numpy(), color="red", label="learned line")
axes[1].set_title("Fitted Line")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 2-1. 🔎 왜 `optimizer.zero_grad()`가 필요한가?

PyTorch는 `.backward()`를 호출할 때마다 기울기를 **누적**합니다.  
초기화하지 않으면 이전 스텝의 기울기가 계속 더해져서 학습이 망가집니다.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = w * 2     # dy/dw = 2
    y.backward()
    print(f"{i+1}번째 backward 후 w.grad = {w.grad.item()}  ← 계속 누적됨")

---
## ✅ 학습 결과 정리 (Verification)

- 수동으로 계산한 미분 값과 PyTorch가 자동으로 계산한 값이 **정확히 일치**함을 확인했다.
- `requires_grad`, `grad_fn`, `backward()`, `torch.no_grad()`의 역할을 직접 확인했다.
- Optimizer가 Gradient 정보를 바탕으로 파라미터를 업데이트하여 Loss가 감소하는 모습을 목격했다.

### 🎯 핵심 결론
딥러닝에서 **학습 = 계산 그래프를 따라 기울기를 흘려보내고, 그 기울기로 파라미터를 조금씩 수정하는 과정**이다.  
Autograd는 이 복잡한 연쇄미분을 자동으로 처리해 주는 엔진이다.